# EXP-BCB-004 — ASTNN trên BigCloneBench (Java) - Kaggle GPU

**Setup Kaggle:** Settings → Accelerator = **GPU (T4/P100)**, Internet = **ON**.

Chạy các cell theo thứ tự. Khi cần tiếp tục phiên bị ngắt, đổi `MODE = 'resume'` ở cell Train.


In [ ]:
import json, os, subprocess, sys

GITHUB_TOKEN = ''
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GITHUB_TOKEN = GITHUB_TOKEN or _s.get_secret('GITHUB_TOKEN')
    HF_TOKEN = HF_TOKEN or _s.get_secret('HF_TOKEN')
except Exception as e:
    print('Dùng token dán trực tiếp nếu có:', e)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
print('GITHUB_TOKEN:', 'OK' if GITHUB_TOKEN else 'TRỐNG', '| HF_TOKEN:', 'OK' if HF_TOKEN else 'TRỐNG')


In [ ]:
REPO = 'kichirou244/SemanticCodeCloneDetection'

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    print(f'$ {cmd}\n{r.stdout}{r.stderr}')
    if r.returncode != 0:
        raise RuntimeError(f'Lệnh thất bại (rc={r.returncode}): {cmd}')
    return r.stdout

if os.path.isdir('project'):
    print('project đã tồn tại -> git pull')
    sh('git -C project pull --ff-only')
else:
    url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git' if GITHUB_TOKEN else f'https://github.com/{REPO}.git'
    sh(f'git clone "{url}" project')
sh(f'git -C project remote set-url origin https://github.com/{REPO}.git')
os.chdir('project')
print('cwd =', os.getcwd())


In [ ]:
sh('pip install -q transformers datasets scikit-learn tree-sitter tree-sitter-java pyyaml')
import torch
assert torch.cuda.is_available(), 'Cần bật GPU trong Settings -> Accelerator -> GPU'
print('GPU:', torch.cuda.get_device_name(0), '| VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')


In [ ]:
# Chuẩn bị dữ liệu BigCloneBench (tải tập smoke và dev qua streaming)
import os
if os.path.exists('/kaggle/input/bigclonebench-features/bcb_ast_cache.npz'):
    print('Phát hiện Kaggle Dataset: Đã có sẵn AST cache!')
    os.makedirs('artifacts/bcb_features', exist_ok=True)
    sh('cp /kaggle/input/bigclonebench-features/bcb_ast_cache.npz artifacts/bcb_features/')
sh('python scripts/download_bcb_subsets.py --smoke --dev')
sh('ls -lh data/raw/bigclonebench/')


In [ ]:
# Preflight check: kiểm tra cấu hình configs/exp004_astnn_bcb.yaml
sh('python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode preflight --device cuda')


In [ ]:
# Diagnostic check: kiểm tra 1 batch loss giảm
sh('python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode diagnostic --device cuda')


In [ ]:
# Huấn luyện mô hình (hỗ trợ resume và time budget)
MODE = 'train'  # 'train' hoặc 'resume'
TIME_BUDGET_S = 9 * 3600  # Giới hạn 9h an toàn
sh(f'python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode {MODE} --device cuda --time-budget-s {TIME_BUDGET_S}')


In [ ]:
# Đánh giá trên tập test và chốt metrics
sh('python -m src.experiments.bcb_runner --config configs/exp004_astnn_bcb.yaml --mode evaluate --device cuda')


In [ ]:
# Đóng gói kết quả metrics và predictions để tải về (dung lượng nhẹ vài MB)
sh('tar -czf exp-bcb-004_results.tar.gz results/')
sh('ls -lh exp-bcb-004_results.tar.gz')
